In [ ]:
# =========================
# IMPORTS
# =========================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris, load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.tree import (
    DecisionTreeClassifier,
    DecisionTreeRegressor,
    plot_tree
)
from sklearn.metrics import accuracy_score, mean_squared_error

# =========================
# PART A: CLASSIFICATION (IRIS DATASET)
# =========================

# Load Iris dataset
iris = load_iris()
X_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
y_iris = pd.Series(iris.target, name="species")

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_iris, y_iris, test_size=0.2, random_state=42
)

# ---- Gini Index ----
clf_gini = DecisionTreeClassifier(
    criterion="gini",
    random_state=42
)
clf_gini.fit(X_train, y_train)
y_pred_gini = clf_gini.predict(X_test)
print("Iris Classification Accuracy (Gini):",
      accuracy_score(y_test, y_pred_gini))

# ---- Entropy ----
clf_entropy = DecisionTreeClassifier(
    criterion="entropy",
    random_state=42
)
clf_entropy.fit(X_train, y_train)
y_pred_entropy = clf_entropy.predict(X_test)
print("Iris Classification Accuracy (Entropy):",
      accuracy_score(y_test, y_pred_entropy))

# ---- Pre-Pruning ----
clf_pruned = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=3,
    min_samples_split=5,
    random_state=42
)
clf_pruned.fit(X_train, y_train)
print("Iris Accuracy (Pre-Pruned Tree):",
      accuracy_score(y_test, clf_pruned.predict(X_test)))

# ---- Post-Pruning (Cost Complexity) ----
path = clf_entropy.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas

clf_post = DecisionTreeClassifier(
    ccp_alpha=ccp_alphas[len(ccp_alphas)//2],
    random_state=42
)
clf_post.fit(X_train, y_train)
print("Iris Accuracy (Post-Pruned Tree):",
      accuracy_score(y_test, clf_post.predict(X_test)))

# ---- Visualization ----
plt.figure(figsize=(18, 8))
plot_tree(
    clf_pruned,
    feature_names=iris.feature_names,
    class_names=iris.target_names,
    filled=True
)
plt.title("Decision Tree (Iris - Pre-Pruned)")
plt.show()

# ---- Feature Importance ----
iris_importance = pd.Series(
    clf_pruned.feature_importances_,
    index=iris.feature_names
).sort_values(ascending=False)

print("\nIris Feature Importance:")
print(iris_importance)

# =========================
# PART B: REGRESSION (DIABETES DATASET)
# =========================

# Load Diabetes dataset
diabetes = load_diabetes()
X_diabetes = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
y_diabetes = pd.Series(diabetes.target, name="disease_progression")

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_diabetes, y_diabetes, test_size=0.2, random_state=42
)

# ---- Unpruned Regressor ----
reg = DecisionTreeRegressor(random_state=42)
reg.fit(X_train, y_train)
y_pred = reg.predict(X_test)
print("\nDiabetes MSE (Unpruned):",
      mean_squared_error(y_test, y_pred))

# ---- Pre-Pruning ----
reg_pruned = DecisionTreeRegressor(
    max_depth=4,
    min_samples_split=10,
    random_state=42
)
reg_pruned.fit(X_train, y_train)
print("Diabetes MSE (Pre-Pruned):",
      mean_squared_error(y_test, reg_pruned.predict(X_test)))

# ---- Post-Pruning ----
path = reg.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas

reg_post = DecisionTreeRegressor(
    ccp_alpha=ccp_alphas[len(ccp_alphas)//2],
    random_state=42
)
reg_post.fit(X_train, y_train)
print("Diabetes MSE (Post-Pruned):",
      mean_squared_error(y_test, reg_post.predict(X_test)))

# ---- Feature Importance ----
diabetes_importance = pd.Series(
    reg_pruned.feature_importances_,
    index=diabetes.feature_names
).sort_values(ascending=False)

print("\nDiabetes Feature Importance:")
print(diabetes_importance)

# ---- Visualization ----
plt.figure(figsize=(20, 10))
plot_tree(
    reg_pruned,
    feature_names=diabetes.feature_names,
    filled=True
)
plt.title("Decision Tree Regressor (Diabetes - Pre-Pruned)")
plt.show()
